# Session 2 · Day 1 - Put TESDA Centers on a Road Network and Measure Access to Programs

Three real files describe TESDA's institutions and what they offer. The **institution master**
(`tesda_coordinates.parquet`) carries checked coordinates and PSGC codes. The **Accredited Assessment
Center** list (CSV) carries which qualifications a center may assess. The **Compendium of Registered
Programs** (Excel) carries which training programs a provider runs. This lab loads the three files,
turns the institutions into map points, drops them on a real road network from OpenStreetMap, and
measures travel to a program three ways: in a straight line, by road, and on foot. The last section
reads how reachable each NCR district is, which is where Day 2 picks up. *Modeling is Session 3.*

### What this notebook covers
1. Load the three source files and read each one's grain
2. Build the located institutions as a `GeoDataFrame`, and round-trip them through GeoJSON
3. Narrow to NCR and choose a center to measure from
4. Build a drivable road network around the center from OpenStreetMap
5. Compare straight-line, drive, and walk distance to the nearby centers
6. Join the registered programs to the NCR centers on institution name
7. Read program access across NCR by district
8. Spot a counting flaw, map one qualification, and work a team scoping activity

> **How to use this notebook**
> - ▶ **Run each cell in order** (`Shift+Enter`). The lecture advances one small step per cell, and later cells depend on earlier ones.
> - 🌐 **Two cells reach the internet** beyond the data bucket: the OpenStreetMap downloads in the road-network section. Each takes a few seconds to a few minutes.
> - ✏️ **Exercise cells** have `### START CODE HERE ###` … `### END CODE HERE ###`. Write code between those lines only.
> - ✅ After an exercise, run the **check cell**. Fix the code until it prints a pass.
> - 🎛️ **Play-around cells** are open to tweak: change a value, re-run, compare.
>
> _Real data loads over the network from a public bucket. No Google Drive mount and no local paths are needed._

### New to Python or pandas?

Anyone coming from another language can run `S2_commands_primer` first. It lays out the mental model
in one page: a `DataFrame` is a table (a SQL table, an R `data.frame`, a spreadsheet), a `merge` is a
`JOIN`, `NaN` is the missing value, and a chain of `.method().method()` reads left to right. The
comments in the code cells point out each pandas idiom as it appears.

In [ ]:
# Run once per session (skip if already installed locally).
!pip -q install geopandas shapely pyproj folium mapclassify openpyxl pyarrow osmnx networkx

In [ ]:
import os
import time
import tempfile
import warnings
from urllib.parse import quote

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
import osmnx as ox
import networkx as nx

warnings.filterwarnings("ignore")                  # the Excel reader warns about a few impossible dates
ox.settings.log_console = False
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# Public TESDA bucket. Spaces in a filename are percent-encoded for an HTTP read.
BASE = "https://storage.googleapis.com/tesda-datasets/data/"

def source(name):
    return BASE + quote(name)

print("osmnx", ox.__version__, "· networkx", nx.__version__, "· geopandas", gpd.__version__)

In [ ]:
# ------------------------------------------------------------------
# Optional: load from Google Drive instead of the public bucket.
# For anyone who already has the TESDA dataset in their Google Drive.
# Uncomment, mount the Drive, and point BASE at the folder with the files.
# ------------------------------------------------------------------
# from google.colab import drive
# drive.mount("/content/drive")
# BASE = "/content/drive/MyDrive/TESDA/data/"   # <- the dataset folder (end with a slash)
# def source(name):                              # local paths need no percent-encoding
#     return BASE + name

## Load the three source files and read each one's grain

Each file lands in its own `DataFrame`, and the first question of any table is what one row means.
The institution master comes first. One row is one institution as recorded in a source list.

In [ ]:
institutions_df = pd.read_parquet(source("tesda_coordinates.parquet"))

In [ ]:
institutions_df.shape   # (rows, columns)

**Expected output:** `(8007, 21)`. The master stacks training providers and assessment centers
into one table of 8007 rows and 21 columns.

In [ ]:
institutions_df.head(3)

Each row carries a `tesda_inst_id`, a name, a PSGC-aligned address, `latitude`/`longitude`, and a
`coord_status` flag that marks whether the pin passed the Day 1 coordinate audit. A `valid` flag is
the signal that a row can go on a map.

In [ ]:
valid = institutions_df["coord_status"] == "valid"
print("valid pins:", int(valid.sum()), "of", len(institutions_df))

**Expected output:** `valid pins: 7891 of 8007`. The remaining 116 rows carry a flagged coordinate
(blank, out of the country, or at sea) and stay off the map until repaired.

The Compendium of Registered Programs is an Excel workbook. Reading every column as text with
`dtype=str` stops pandas from silently guessing a type, and the contact and trainer columns are
dropped at the door as personal data.

In [ ]:
compendium_df = pd.read_excel(
    source("Compendium of Registered Programs as of June 2026.xlsx"),
    sheet_name="Compendium ao June 2026",
    dtype=str,                   # read every value as text
)

In [ ]:
compendium_df = compendium_df.drop(columns=["TEL. NO.", "E-mail Address", "Name of Trainer/s", "NTTC No."])

In [ ]:
compendium_df.shape

**Expected output:** `(21967, 19)`. One row is one registered program at one provider, so a provider
that runs ten programs appears on ten rows.

The Accredited Assessment Center list is a CSV. The same `dtype=str` applies, `utf-8-sig` drops the
byte-order mark some exports add, `str.strip()` trims stray spaces, and the two personal-data columns
are dropped. Each step is its own cell so the table can be inspected after each change.

In [ ]:
accredited_centers_df = pd.read_csv(
    source("2026 Accredited Compentency Assessment Center based on July 15, 2026 database.csv"),
    dtype=str,
    encoding="utf-8-sig",        # utf-8-sig drops the BOM some exports add
)

In [ ]:
accredited_centers_df = accredited_centers_df.apply(lambda s: s.str.strip())   # .str runs a string op down a column

In [ ]:
accredited_centers_df = accredited_centers_df.drop(columns=["Center Manager", "Phone"])   # personal data, never kept

In [ ]:
accredited_centers_df.shape

**Expected output:** `(12871, 14)`. One row is one accreditation: a center accredited for ten
qualifications appears on ten rows.

In [ ]:
accredited_centers_df.head(3)

## Build the located institutions as a GeoDataFrame

A `GeoDataFrame` is a `DataFrame` with one extra column, `geometry`, holding a shape per row. Here the
shape is a point built from `longitude` and `latitude`. Blanking the flagged pins with `.where(valid)`
leaves the bad rows with an empty geometry instead of a point in the wrong place.

In [ ]:
institutions_gdf = gpd.GeoDataFrame(
    institutions_df.copy(),
    # points_from_xy builds a point column from lon/lat, and .where(valid) blanks the bad pins
    geometry=gpd.points_from_xy(institutions_df["longitude"].where(valid), institutions_df["latitude"].where(valid)),
    crs="EPSG:4326",             # WGS 84: plain longitude/latitude in degrees
)

In [ ]:
institutions_gdf[["tesda_inst_id", "name", "region", "geometry"]].head()

The `geometry` column now reads `POINT (lon lat)`. A quick scatter of every located pin shows the
archipelago, and a stray point in the sea or abroad would stand out at this zoom.

In [ ]:
institutions_gdf.plot(markersize=0.5, figsize=(6, 9))
plt.title(f"{int(valid.sum()):,} located TESDA institutions")
plt.show()

## Write and read the located pins as GeoJSON

GeoJSON is the plain-text interchange format for map data, readable by almost any GIS tool. Writing
only the located pins and reading them back confirms the geometry survives a round-trip through a
file, the same way a shapefile or GeoPackage would carry it.

In [ ]:
WORK = tempfile.mkdtemp(prefix="s2d1_06_")           # one temp folder for files written here
geojson_path = os.path.join(WORK, "tesda_institutions.geojson")
institutions_gdf[valid].to_file(geojson_path, driver="GeoJSON")   # only the located pins

In [ ]:
institutions_geojson = gpd.read_file(geojson_path)
print("read", institutions_geojson.shape[0], "features from GeoJSON, CRS", institutions_geojson.crs)

**Expected output:** `read 7891 features from GeoJSON, CRS EPSG:4326`. Only the located pins were
written, so the feature count equals the valid-pin count.

## Narrow to NCR and choose a center to measure from

The road-network work stays inside one region so the download is small and the maps are readable.
`query` filters rows with a readable string, the same job as a boolean mask `df[df["region"] == "NCR"]`.

In [ ]:
ncr_gdf = institutions_gdf.query("region == 'NCR'").copy()
ncr_gdf.shape

**Expected output:** `(933, 22)`. The 22nd column is `geometry`, added when the `GeoDataFrame` was
built.

In [ ]:
ncr_located = ncr_gdf[ncr_gdf["coord_status"] == "valid"].copy()
print("NCR institutions:", len(ncr_gdf), "| located:", len(ncr_located))

**Expected output:** `NCR institutions: 933 | located: 919`. Fourteen NCR rows carry no usable pin and
cannot be placed on the network.

The center to measure from is SEAMEO INNOTECH on Commonwealth Avenue, Quezon City, the same landmark
anchor used in `S2_D1_01`. Its coordinates are fixed as constants so every distance below is measured
from one agreed point.

In [ ]:
CENTER_NAME = "SEAMEO INNOTECH, Quezon City"
CENTER_LAT, CENTER_LON = 14.6601097, 121.0565753
print("measuring from:", CENTER_NAME, "at", CENTER_LAT, CENTER_LON)

Reading a single institution's coordinates is a common need, for example to center a map on one
center. Selecting the row by its id and taking `.geometry` returns a point whose `.x` is longitude and
`.y` is latitude, in that order.

In [ ]:
one = institutions_gdf.query("tesda_inst_id == 'TESDA00805'")
point = one.geometry.iloc[0]
print(one["name"].iloc[0])
print("longitude:", round(point.x, 5), "| latitude:", round(point.y, 5))

A point stores `x` then `y`, so longitude comes before latitude. Mapping libraries expect the pair in
different orders (folium wants `[lat, lon]`, a shapely `Point` wants `(lon, lat)`), and a swap there is
the most common mapping bug. Naming the values on the way out avoids it.

## Build a drivable road network around the center

OpenStreetMap publishes the road geometry of the whole world, and OSMnx downloads a piece of it as a
routable graph: intersections become nodes, road segments become edges. `graph_from_point` fetches
every drivable road within a radius of the center. `simplify=True` folds a chain of nodes along one
road into a single edge.

Two steps reach the internet beyond the data bucket: the drive-network download here and the walk
network later. Each takes a few seconds to a few minutes.

In [ ]:
AREA_NAME = "Quezon City around SEAMEO INNOTECH"
RADIUS_M = 4000               # metres around the center point
RADIUS_KM = RADIUS_M / 1000

start = time.time()
G_drive = ox.graph.graph_from_point(
    (CENTER_LAT, CENTER_LON),
    dist=RADIUS_M,
    network_type="drive",     # car-legal roads only
    simplify=True,
)
print(f"drive network for {AREA_NAME}: "
      f"{len(G_drive):,} nodes, {G_drive.number_of_edges():,} edges "
      f"(in {time.time() - start:.0f} s)")

**Expected output:** a drive network of roughly `6,118 nodes, 14,647 edges` (the exact counts shift as
OpenStreetMap is edited). Splitting the graph into two `GeoDataFrame`s, one of nodes and one of edges,
makes it a table again, ready to measure and draw.

In [ ]:
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G_drive, nodes=True, edges=True)

In [ ]:
edges_gdf[["name", "highway", "oneway", "length", "geometry"]].head()

Each edge carries a `highway` tag naming its road class (a motorway, a residential street, a service
road). Some tags arrive as a list when one segment carries several values, so `first_value` keeps the
first, and `_link` ramps fold into their parent class.

In [ ]:
def first_value(v):
    """Some OSM tags hold a list when one segment carries several values. Keep the first."""
    if isinstance(v, list):
        return v[0]
    return v

edges_gdf["road_class"] = edges_gdf["highway"].map(first_value)                   # .map runs first_value down the column
edges_gdf["road_class"] = edges_gdf["road_class"].str.replace("_link", "", regex=False)

In [ ]:
total_km = edges_gdf["length"].sum() / 1000
named = edges_gdf.loc[edges_gdf["name"].notna(), "name"].map(first_value).nunique()
print(f"{len(edges_gdf):,} road segments · {total_km:,.1f} km of drivable road · {named:,} distinct road names")

**Expected output:** about `14,647 road segments · 1,404.1 km of drivable road · 1,779 distinct road
names` for this focus area. The length by class and a map of the network put the hierarchy side by
side: a few long arterials carry most of the distance, with the center marked.

In [ ]:
ORDER = ["motorway", "trunk", "primary", "secondary", "tertiary",
         "unclassified", "residential", "living_street", "service"]
COLORS = {"motorway": "#b2182b", "trunk": "#ef8a62", "primary": "#f4a582", "secondary": "#2166ac",
          "tertiary": "#67a9cf", "unclassified": "#999999", "residential": "#bbbbbb",
          "living_street": "#cccccc", "service": "#dddddd", "other": "#eeeeee"}
WIDTH = {"motorway": 2.2, "trunk": 1.8, "primary": 1.3, "secondary": 1.0, "tertiary": 0.7}

km = edges_gdf.groupby("road_class")["length"].sum() / 1000   # metres per class to km (SQL GROUP BY + SUM)
km_ord = km.reindex(ORDER).dropna()                           # force the hierarchy order
other = km[~km.index.isin(ORDER)].sum()                       # ~isin = NOT IN: classes outside the named set
if other > 0:
    km_ord["other"] = other

In [ ]:
from shapely.geometry import Point

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1, 1.4]})

vals = km_ord.values[::-1]            # [::-1] reverses, so the longest class sits at the top
labs = list(km_ord.index[::-1])
a1.barh(labs, vals, color=[COLORS.get(c, "#dddddd") for c in labs])
for y_idx, v in enumerate(vals):
    a1.text(v, y_idx, f" {v:,.1f} km", va="center", fontsize=8)
a1.set_xlim(0, max(vals) * 1.22)
a1.set_title("Drivable road length by class")
a1.set_xlabel("km")

edges_3857 = edges_gdf.to_crs(3857)   # Web Mercator, so the map draws on a flat axis
for cls in ORDER[::-1]:
    sub = edges_3857[edges_3857["road_class"] == cls]
    if len(sub):
        sub.plot(ax=a2, color=COLORS[cls], linewidth=WIDTH.get(cls, 0.4), label=cls)

marker_3857 = gpd.GeoDataFrame(geometry=[Point(CENTER_LON, CENTER_LAT)], crs="EPSG:4326").to_crs(3857)
marker_3857.plot(ax=a2, color="red", marker="*", markersize=200, zorder=5, label="center")
a2.set_axis_off()
a2.set_title(f"Drivable network around {AREA_NAME}")
a2.legend(loc="lower left", fontsize=7, frameon=True)
plt.tight_layout()
plt.show()

## Compare straight-line, drive, and walk distance

How far is a center? It depends on how the distance is measured. Three ways give three different
numbers, and the gaps between them are the whole point here:

| Way to measure | What it counts | Usually |
|---|---|---|
| straight line | the direct distance between two points | **too low**, since it ignores roads |
| drive | the shortest route on car roads | longer, because of one-way streets and rivers to cross |
| walk | the shortest route on footpaths and small streets | short in metres but slow in time |

The straight-line distance needs no road data at all. Latitude and longitude are in degrees, which do
not measure metres evenly, so the points are reprojected to a metres-based system (`EPSG:3125`, meant
for the Philippines). After that, `.distance` returns metres.

In [ ]:
center_metric = gpd.points_from_xy([CENTER_LON], [CENTER_LAT], crs="EPSG:4326").to_crs(3125)[0]
ncr_located["dist_straight_km"] = ncr_located.to_crs(3125).geometry.distance(center_metric) / 1000

In [ ]:
ncr_located.sort_values("dist_straight_km")[["name", "dist_straight_km"]].head(5).round(2)

**Expected output:** the five nearest centers by straight line, from about `0.21 km` to `1.67 km`,
led by `Mother Theresa Institute of Technology and Assessment Center`. These straight-line figures set
the floor the road and walk distances are compared against.

The road and walk distances need a network each. The walk network downloads the same way, with
`network_type="walk"` so it includes footpaths a car cannot use.

In [ ]:
start = time.time()
G_walk = ox.graph.graph_from_point(
    (CENTER_LAT, CENTER_LON),
    dist=RADIUS_M,
    network_type="walk",
    simplify=True,
)
print(f"walk network: {len(G_walk):,} nodes, {G_walk.number_of_edges():,} edges "
      f"(in {time.time() - start:.0f} s)")

Dijkstra's algorithm, run once from the center, finds the shortest distance to every other node in the
network in a single pass. Ranking five nearby centers or five hundred costs the same one search. The
center snaps to its nearest road node, and so does each nearby institution. Only the institutions
inside the downloaded radius are measured, since anything farther has no road node here to reach.

In [ ]:
origin_drive = ox.distance.nearest_nodes(G_drive, CENTER_LON, CENTER_LAT)
origin_walk = ox.distance.nearest_nodes(G_walk, CENTER_LON, CENTER_LAT)

# shortest distance in metres from the center to every reachable node
drive_m = nx.single_source_dijkstra_path_length(G_drive, origin_drive, weight="length")
walk_m = nx.single_source_dijkstra_path_length(G_walk, origin_walk, weight="length")

In [ ]:
near = ncr_located[ncr_located["dist_straight_km"] <= RADIUS_KM].copy()
near["node_drive"] = ox.distance.nearest_nodes(G_drive, X=near["longitude"].values, Y=near["latitude"].values)
near["node_walk"] = ox.distance.nearest_nodes(G_walk, X=near["longitude"].values, Y=near["latitude"].values)
near["dist_drive_km"] = near["node_drive"].map(drive_m) / 1000
near["dist_walk_km"] = near["node_walk"].map(walk_m) / 1000
print(f"{len(near)} located NCR centers within {RADIUS_KM:.0f} km straight-line of the center, routed on both networks")

Turning distance into time needs a speed per ruler. A city drive averages far below the posted limit
with traffic and stops, and a steady walk is about 4.5 km/h.

In [ ]:
DRIVE_SPEED_KMH = 30.0
WALK_SPEED_KMH = 4.5

near["time_drive_min"] = near["dist_drive_km"] / DRIVE_SPEED_KMH * 60
near["time_walk_min"] = near["dist_walk_km"] / WALK_SPEED_KMH * 60

show = ["name", "dist_straight_km", "dist_drive_km", "dist_walk_km", "time_drive_min", "time_walk_min"]
near.sort_values("dist_straight_km")[show].head(5).round(2)

**Expected output:** a five-row table where `dist_drive_km` is larger than `dist_straight_km` for most
rows (for example a center 0.92 km away in a straight line is about 1.04 km by road), and
`time_walk_min` is far bigger than `time_drive_min`. The exact metres depend on the live OpenStreetMap
download, so treat these as representative. How much longer the road route is than the straight line is
a handy signal, and Day 2 turns it into a model input.

## Join the registered programs to the NCR centers

A center's location says where it is, and the Compendium says what it teaches. Joining the two attaches
the program list to each NCR institution. The files share no id, so the join rests on the institution
name, lower-cased and trimmed so small spelling differences line up.

In [ ]:
ncr_gdf["clean_name"] = ncr_gdf["name"].str.strip().str.lower()
compendium_df["clean_name"] = compendium_df["NAME OF INSTITUTION"].str.strip().str.lower()

In [ ]:
ncr_programs = ncr_gdf.merge(compendium_df, on="clean_name", suffixes=("_inst", "_comp"))
print("program rows joined to NCR institutions:", len(ncr_programs))

**Expected output:** `program rows joined to NCR institutions: 3546`. A name-only join is quick but
approximate: it can miss a center whose name is spelled differently in the two files, and it can link a
program from a same-named provider in another province. A name-and-address key, or a shared institution
id, would be more reliable. The link rate is worth reporting whenever a name-based join feeds an
analysis.

In [ ]:
ncr_programs["PROGRAM"].value_counts().head(8)

**Expected output:** the most common NCR programs, led by `Bread and Pastry Production NC II` (about
235 rows), `Computer Systems Servicing NC II`, and `Housekeeping NC II`. These are program rows, so one
institution contributes several.

## Read program access across NCR by district

Access has two parts: **supply**, how many located institutions a district has, and **proximity**, how
far they sit from the center being measured. One `groupby` on district reads both at once (same idea as
a SQL `GROUP BY`). The median distance is used instead of the mean, so a single mislocated pin does not
throw off a district's number.

In [ ]:
mm = ncr_located[ncr_located["province"].str.startswith("Metropolitan Manila", na=False)]
access = (mm.groupby("province")
            .agg(located_institutions=("name", "size"),
                 median_km_from_center=("dist_straight_km", "median"))
            .sort_values("located_institutions", ascending=False)
            .round(2))
access

**Expected output:** four Metro Manila districts, led by the **Second District** with about **314**
located institutions at a median of roughly **6 km** from the center, then the Fourth District (about
305, around 16 km), the First (about 152, around 11 km), and the Third (about 133, around 10 km). The
center sits inside the Second District, so that district comes out both the densest and the nearest.
The southern Fourth District has almost as many institutions but at more than twice the distance. Many
institutions that are still far away is the kind of gap Day 2 goes after with the two-step floating
catchment area (2SFCA).

## ✏️ Exercise - count centers within reach

Count the **located NCR institutions within 5 km straight-line** of the center. The straight-line
distance already sits in `ncr_located["dist_straight_km"]`. Store the integer in `n_within_5km`.

**Hint:** build a boolean mask `ncr_located["dist_straight_km"] <= 5`, then take its `.sum()` and wrap
it in `int(...)`.

In [ ]:
### START CODE HERE ###   (about 1 line)
n_within_5km = None   # located NCR institutions within 5 km straight-line of the center
### END CODE HERE ###
print("located NCR institutions within 5 km:", n_within_5km)

## ✅ Check the work

Run this. It recomputes the reference and compares it to `n_within_5km`.

In [ ]:
try:
    ref = int((ncr_located["dist_straight_km"] <= 5).sum())
    assert n_within_5km is not None, "n_within_5km is None, fill in the code."
    assert n_within_5km == ref, f"expected {ref}, got {n_within_5km}"
    print(f"✅ Passed! {ref} located NCR institutions sit within 5 km of the center.")
except AssertionError as e:
    print("❌ Not yet:", e)

## ✏️ Exercise - count centers for one qualification

Using the joined table `ncr_programs`, count the **distinct NCR institutions** whose `PROGRAM` mentions
**Cookery**. Store the integer in `n_cookery_ncr`.

**Hint:** filter with `ncr_programs["PROGRAM"].str.contains("Cookery", case=False, na=False)`, then
count distinct institutions with `["name"].nunique()`.

In [ ]:
### START CODE HERE ###   (about 2 lines)
n_cookery_ncr = None   # distinct NCR institutions offering a Cookery program
### END CODE HERE ###
print("NCR institutions offering Cookery:", n_cookery_ncr)

## ✅ Check the work

Run this. It recomputes the reference and compares it to `n_cookery_ncr`.

In [ ]:
try:
    ref_cook = ncr_programs[ncr_programs["PROGRAM"].str.contains("Cookery", case=False, na=False)]
    ref = int(ref_cook["name"].nunique())
    assert n_cookery_ncr is not None, "n_cookery_ncr is None, fill in the code."
    assert n_cookery_ncr == ref, f"expected {ref}, got {n_cookery_ncr}"
    print(f"✅ Passed! {ref} distinct NCR institutions offer a Cookery program.")
except AssertionError as e:
    print("❌ Not yet:", e)

### ❓ Question

Read the valid-pin count printed when the master loaded. How many institution rows carry a usable
coordinate (`coord_status == "valid"`)?

In [ ]:
q_valid_pins = None   # <- answer here (an integer)

In [ ]:
ref_valid_pins = int((institutions_df["coord_status"] == "valid").sum())
if q_valid_pins == ref_valid_pins:
    print(f"✅ Correct! {ref_valid_pins} rows carry a valid pin.")
else:
    print(f"❌ Not yet: expected {ref_valid_pins}, got {q_valid_pins}.")

### ❓ Question

Read the join report. How many **program rows** joined to the NCR institutions on name?

In [ ]:
q_ncr_programs = None   # <- answer here (an integer)

In [ ]:
ref_ncr_programs = len(ncr_programs)
if q_ncr_programs == ref_ncr_programs:
    print(f"✅ Correct! {ref_ncr_programs} program rows joined to NCR institutions.")
else:
    print(f"❌ Not yet: expected {ref_ncr_programs}, got {q_ncr_programs}.")

## 🐞 Spot the flaw - reading the row count as an institution count

A quick headline: "NCR has 3546 program offerings, so about 3546 training slots". Reading the joined
row count as a count of institutions is the flaw, because one institution contributes one row per
program. Run the comparison, then read the gap.

In [ ]:
program_rows = len(ncr_programs)
distinct_institutions = ncr_programs["name"].nunique()
print("joined program rows:          ", program_rows)
print("distinct institutions:        ", distinct_institutions)
print("programs per institution:     ", round(program_rows / distinct_institutions, 1))

**Expected output:** about `3546` rows across roughly `649` distinct institutions, near `5.5` programs
each. The row count answers "how many program offerings". The distinct count answers "how many
institutions". Treating one as the other inflates the number of places a learner can actually go. When
the question is about institutions, count distinct ids instead of rows.

## 🎛️ Play around - change the reach radius

Pick a radius and count the located NCR institutions within it in a straight line. Change `REACH_KM`
and re-run to feel how quickly supply grows with distance. A slider is wired up for Colab or Jupyter,
with a plain fallback everywhere else.

In [ ]:
REACH_KM = 5.0   # try 2, 3, 8, 10

def centers_within(reach_km):
    n = int((ncr_located["dist_straight_km"] <= reach_km).sum())
    print(f"{n} located NCR institutions within {reach_km:.0f} km straight-line of the center")

centers_within(REACH_KM)

try:
    import ipywidgets as widgets
    from IPython.display import display
    slider = widgets.FloatSlider(value=REACH_KM, min=1, max=20, step=1, description="reach km")
    widgets.interact(centers_within, reach_km=slider)
except Exception:
    for r_try in [2, 5, 10, 20]:
        centers_within(r_try)

## 🤝 Team activity - a brief for one qualification

Work this one as a team, rotating roles. The joined table and the distances already computed hold
everything the question needs.

| Role | Does |
|---|---|
| Driver | types the code |
| Navigator | reads the steps, checks each output |
| Validator | spot-checks two or three centers against a public map |
| Scribe | writes the answer and the reasoning |

**The scenario.** A regional office wants a one-page brief for one qualification (choose one, for
example `Cookery NC II` or `Shielded Metal Arc Welding`) across NCR.

Produce three things:

1. the number of **distinct located** NCR institutions whose program matches the qualification,
2. the five **cities** holding the most such institutions (`City Name` in the Compendium, or
   `city_municipality` on the institution side), and
3. one sentence on how a learner's **access** to this qualification differs by district, using the
   supply-and-proximity table built above.

**Scribe's notes.** Record the qualification chosen, the three results, and one caveat about the
name-only join (how many matches might be a same-named provider elsewhere, and how a spot-check would
confirm them).

## Preview the barangay boundaries for later aggregation

Counts by district used the recorded `province` field. Day 2 instead counts points inside real
**barangay polygons** from a GeoPackage, so a pin lands in a barangay by its actual location,
independent of how its row was labeled. GDAL's `/vsicurl/` driver opens the big file in place over HTTP and
reads only the byte ranges a few rows touch, so a preview costs a few requests. The full treatment is
in `S2_D1_02`.

In [ ]:
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".gpkg"
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
VSI = "/vsicurl/" + BASE

brgy_gpkg = gpd.read_file(VSI + "consolidated_geodata_matched.gpkg", rows=100)
print("read", brgy_gpkg.shape[0], "barangay boundaries, CRS", brgy_gpkg.crs)
brgy_gpkg[["name", "adm1_en", "adm2_en", "adm3_en", "geometry"]].head()

**Expected output:** `read 100 barangay boundaries, CRS EPSG:4326`, then five rows with a
`MULTIPOLYGON` geometry per barangay. Day 2 joins the located institutions into these polygons to count
and normalize by area.

## Recap
- Three source files load into three tables: the institution master (8007 rows), the Compendium of
  registered programs (21,967 rows), and the accredited assessment centers (12,871 rows).
- Only the **7,891 located pins** go on a map. A `GeoDataFrame` holds the points, and saving them to
  GeoJSON and reading them back shows the geometry travels with the rows.
- Distance depends on how it is measured: the straight line reads **too low**, the road route is
  longer, and a walk is short in metres but slow in time. One Dijkstra search from the center ranks
  every nearby center at once.
- A name-only join attaches **3546** program rows to NCR institutions. It is quick but approximate, so
  the match rate matters, and rows are not the same as institutions.
- Access has two parts: **supply** (institutions per district) and **proximity** (median distance).
  Day 2 combines them with the two-step floating catchment area.

**For discussion**
1. When would the straight-line distance mislead a coverage claim the most (a river, a limited-access
   expressway, a coastline)?
2. The name-only join links about 3546 program rows. What would a name-and-address key, or a shared
   institution id, change about the count and its trustworthiness?
3. District counts used the recorded `province`. How could a point-in-polygon count on real barangay
   boundaries change the picture for a district with mislocated pins?

**Next:** `S2_D1_02` for the admin-boundary joins previewed here, and `S2_D2_02` for the accessibility
and 2SFCA that this access read leads into.

## References & further reading

- [pandas, user guide](https://pandas.pydata.org/docs/user_guide/)
- [pandas, merge and join](https://pandas.pydata.org/docs/user_guide/merging.html)
- [GeoPandas, user guide & API](https://geopandas.org/)
- [folium (interactive maps)](https://python-visualization.github.io/folium/)
- [OSMnx (street networks from OpenStreetMap)](https://osmnx.readthedocs.io/)
- [NetworkX, shortest paths](https://networkx.org/documentation/stable/reference/algorithms/shortest_paths.html)
- [OpenStreetMap (the road data)](https://www.openstreetmap.org/)
- [PSA, Philippine Standard Geographic Code (PSGC)](https://psa.gov.ph/classification/psgc)